In [ ]:
from pathlib import Path
import h5py
import numpy as np
import pandas as pd
import openslide
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = ROOT / "data"
FEAT = ROOT / "features/lunit_vits16"
manifest = pd.read_csv(DATA / "manifest.csv", dtype={"patient_id": str})

Checking that every feature file matches its tiling file

In [ ]:
rows = []
for r in manifest.itertuples():
    with h5py.File(ROOT / "features/tiles" / r.cohort / f"{r.patient_id}.h5") as t, \
         h5py.File(FEAT / r.cohort / f"{r.patient_id}.h5") as f:
        feats = f["features"][:].astype(np.float32)
        rows.append({"patient_id": r.patient_id, "cohort": r.cohort,
                     "n_tiles": len(t["coords"]), "n_feats": len(feats),
                     "coords_match": np.array_equal(t["coords"][:], f["coords"][:]),
                     "all_finite": bool(np.isfinite(feats).all()),
                     "zero_rows": int((np.abs(feats).sum(1) == 0).sum()),
                     "mean_norm": float(np.linalg.norm(feats, axis=1).mean())})
chk = pd.DataFrame(rows)
print((chk.n_tiles == chk.n_feats).all(), chk.coords_match.all(),
      chk.all_finite.all(), chk.zero_rows.sum())
chk.groupby("cohort")["mean_norm"].describe()

Check confirmed that:
- tile counts match feature counts for all slides
- the coordinates are consistent between tiles and features
- there are no missing, infinite values and no empty rows

In [ ]:
# Take 200 random tiles for each slide and load their embeddings
rng = np.random.default_rng(123)
X, meta = [], []
for r in manifest.itertuples():
    with h5py.File(FEAT / r.cohort / f"{r.patient_id}.h5") as f:
        n = len(f["coords"])
        idx = np.sort(rng.choice(n, size=min(200, n), replace=False))
        X.append(f["features"][idx].astype(np.float32))
        meta += [(r.cohort, r.patient_id, r.slide_path, *f["coords"][i]) for i in idx]
X = np.concatenate(X)
# unit length, so dot product = cosine similarity
X /= np.linalg.norm(X, axis=1, keepdims=True)
meta = pd.DataFrame(meta, columns=["cohort", "patient_id", "slide_path", "x", "y"])
print(X.shape)

Nearest neighbours

In [ ]:
def read_tile(row):
    s = openslide.OpenSlide(str(DATA / row.slide_path))
    l0 = round(224 * 0.5 / float(s.properties[openslide.PROPERTY_NAME_MPP_X]))
    tile = s.read_region((int(row.x), int(row.y)), 0, (l0, l0)).convert("RGB")
    s.close()
    return tile

def show_neighbours(q, k=5):
    sims = X @ X[q]
    # exclude the query's own slide
    sims[meta.patient_id == meta.patient_id[q]] = -1
    nn = np.argsort(-sims)[:k]
    fig, axes = plt.subplots(1, k + 1, figsize=(3 * (k + 1), 3))
    for ax, i, title in zip(axes, [q, *nn], ["query", *[f"{sims[i]:.2f}" for i in nn]]):
        ax.imshow(read_tile(meta.iloc[i]))
        ax.set_title(f"{title}\n{meta.cohort[i]} {meta.patient_id[i]}", fontsize=8)
        ax.axis("off")

for q in rng.choice(len(X), size=6, replace=False):
    show_neighbours(q)

The similar tiles all look close to the queries. We need to look at cohort clustering though since yale queries are most similar with exclusively yale tiles.

In [ ]:
rng2 = np.random.default_rng(1)
qs = rng2.choice(len(X), size=2000, replace=False)
coh, pid, k = meta.cohort.to_numpy(), meta.patient_id.to_numpy(), 10

same = []
for q in qs:
    s = X @ X[q]
    s[pid == pid[q]] = -1
    nn = np.argpartition(-s, k)[:k]          # top-k indices without a full sort
    same.append((coh[nn] == coh[q]).mean())
same = np.array(same)

for c in ["impress", "yale"]:
    m = coh[qs] == c
    print(f"{c}: same-cohort neighbours {same[m].mean():.2f}  vs base rate {(coh == c).mean():.2f}")

Cohort clustering is very high.

In [ ]:
def cohort_purity(Xm, n=2000, k=10, seed=1):
    r = np.random.default_rng(seed)
    qs = r.choice(len(Xm), size=n, replace=False)
    same = []
    for q in qs:
        s = Xm @ Xm[q]
        s[pid == pid[q]] = -1
        nn = np.argpartition(-s, k)[:k]
        same.append((coh[nn] == coh[q]).mean())
    same = np.array(same)
    return {c: round(same[coh[qs] == c].mean(), 2) for c in ["impress", "yale"]}

Xs = X.copy()
for c in ["impress", "yale"]:
    m = coh == c
    # z-score each dimension within the cohort
    Xs[m] = (Xs[m] - Xs[m].mean(0)) / (Xs[m].std(0) + 1e-6)   
Xs /= np.linalg.norm(Xs, axis=1, keepdims=True)

print("raw:         ", cohort_purity(X))
print("standardised:", cohort_purity(Xs))

Standardizing removed some of the same cohort effect but it is still far above the base rates. We should keep this in mind as it will likely lead to a drop in performance on the test set (which is from the other cohort).